# 09 · Agents and sandboxing

Agentic workloads run **untrusted code**: an LLM writes Python, and something has to execute
it without trusting it. Union ships two complementary sandboxes for exactly this, plus the
building blocks (tasks, traces, apps) that turn an agent loop into production infrastructure.

**Learning goals**

1. Run arbitrary (LLM-generated) code in an ephemeral, isolated container with
   `flyte.sandbox.create`
2. Run LLM-generated *orchestration* in a Monty workflow sandbox — the
   **programmatic tool calling / code mode** pattern
3. Assemble a minimal ReAct agent from primitives you already know
   (`@env.task` + `@flyte.trace`)
4. Know where the rest of the agent stack lives (HITL, memory, chat UI, frameworks)

> ⚠️ **Experimental:** `flyte.sandbox` is alpha — APIs may change between SDK releases.
> Pin your SDK version (these notebooks pin 2.5.7) and re-verify on upgrade.

In [ ]:
import flyte
import flyte.sandbox

flyte.init_from_config()

## 1. Why sandbox LLM-generated code

The model has no intent, but it can still emit `DELETE FROM orders WHERE 1=1`, an infinite
loop, or code that reads your env vars and posts them somewhere. Running it unsandboxed
means trusting the model never to make those mistakes. Union's two approaches:

| | **Code sandbox** | **Workflow sandbox (Monty)** |
|---|---|---|
| Runtime | Ephemeral Docker container, built on demand, discarded after one run | Rust-based sandboxed Python interpreter, in-process |
| Startup | Seconds (image build is cached) | **Microseconds** |
| Capabilities | Full Python — any package, file I/O, shell | Pure control flow only — **no imports, no I/O, no network** |
| Use for | Arbitrary computation: data crunching, test execution, ETL snippets | LLM-written *orchestration* that calls your registered tools |
| State | Stateless per invocation | Runs alongside the worker process |

## 2. Code sandbox: `flyte.sandbox.create`

**Auto-IO mode** (default): declare typed inputs/outputs, write only the business logic.
Flyte generates the argument parsing, exposes inputs as variables, and captures any variable
whose name matches a declared output. Dependencies are declared per-sandbox and the image is
built through your Image Builder → Artifact Registry path, then cached by content.

In [ ]:
sandbox = flyte.sandbox.create(
    name="double",
    code="result = x * 2",
    inputs={"x": int},
    outputs={"result": int},
)

result = await sandbox.run.aio(x=21)
result

Now the realistic case — the code string came from an LLM, needs a third-party package, and
returns multiple typed outputs. Nothing it does inside the container can touch your data
plane: fresh filesystem, one execution, discarded.

In [ ]:
# Pretend an LLM produced this in response to "compute summary stats for these numbers"
llm_generated = '''
import numpy as np
nums = np.array([float(v) for v in values.split(",")])
mean = float(np.mean(nums))
std = float(np.std(nums))
'''

stats_sandbox = flyte.sandbox.create(
    name="llm-stats",
    code=llm_generated,
    inputs={"values": str},
    outputs={"mean": float, "std": float},
    packages=["numpy"],                 # sandbox image deps — built & cached on demand
    timeout=120,                        # hard stop for runaway code
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)

mean, std = await stats_sandbox.run.aio(values="1,2,3,4,5,6")
print(f"mean={mean} std={std}")

Two more modes when auto-IO doesn't fit:

- **Verbatim** (`auto_io=False`) — run a complete script that manages its own I/O via
  `/var/inputs/<name>` and `/var/outputs/<name>` (good for file-shaped payloads)
- **Command** (`command=[...]`) — run any binary or shell pipeline, e.g. a pytest run over
  generated code

Useful knobs: `system_packages` (apt), `env_vars`, `secrets`, `retries`, `cache="auto"`
(identical code + inputs = cache hit, like any task).

## 3. Workflow sandbox: programmatic tool calling (code mode)

Sequential tool calling routes every intermediate result through the model's context —
token-expensive and slow at each round-trip. **Code mode** flips it: the LLM writes *one
program* that calls your tools; the sandbox executes loops/conditionals/data-shuffling
locally and only the final answer returns to the model.

Union implements this with **Monty**: your tools are ordinary `@env.task` workers (full
containers, full capabilities), and the LLM-generated orchestration runs in an interpreter
that structurally *cannot* import, do I/O, or touch the network — it can only call the tools
you registered.

In [ ]:
env = flyte.TaskEnvironment(
    name="code_mode",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)


# Tools: ordinary tasks, run in their own containers with full capabilities
@env.task
def add(x: int, y: int) -> int:
    return x + y


@env.task
def multiply(x: int, y: int) -> int:
    return x * y


# Static form: sandboxed orchestrator you write yourself
@env.sandbox.orchestrator
def pipeline(n: int) -> int:
    partial = add(n, 5)          # each call pauses Monty, runs the worker container
    return multiply(partial, 2)  # then resumes with the result


run = await flyte.run.aio(pipeline, n=10)
print(run.url)
await run.wait.aio()
run.outputs()

The dynamic form is the agent-facing one — the orchestration **code arrives as a string**
(from an LLM) and the value of the last expression becomes the output:

In [ ]:
# Pretend an LLM wrote this plan in response to "add x and y, then scale it"
llm_plan = '''
partial = add(x, y)
multiply(partial, scale)
'''

compute = flyte.sandbox.orchestrator_from_str(
    llm_plan,
    inputs={"x": int, "y": int, "scale": int},
    output=int,
    tasks=[add, multiply],       # the ONLY capabilities the code has
    name="llm-plan",
)

run = await flyte.run.aio(compute, x=2, y=3, scale=4)
print(run.url)
await run.wait.aio()
run.outputs()      # (2+3)*4 = 20

Notice what you get for free because the plan executed *on Union*: every tool call is a
tracked action (durable, retryable, cached per your task config), the whole plan is
observable in the UI, and inputs/outputs are typed and validated at the sandbox boundary.
That's the difference between this and an `exec()` in your agent process.

## 4. A minimal ReAct agent from primitives you already know

Union is framework-agnostic — an agent loop is just Python inside a task, using any LLM SDK.
The platform's role is the production layer: each agent step is a **sandboxed container**
(`@env.task`), every LLM/tool call is a **traced checkpoint** (`@flyte.trace`, from
[02](./02-parallelism-and-resilience.ipynb) §5), and the loop's state is durable.

The demo below uses a deterministic mock "LLM" so it runs without an API key — swap
`reason()`'s body for a real OpenAI/Anthropic call (inject the key via `flyte.Secret`,
[05](./05-gcp-data-and-integrations.ipynb) §3) and nothing else changes.

In [ ]:
import json

agent_env = flyte.TaskEnvironment(
    name="react_agent",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    # For a real LLM: secrets=[flyte.Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY")]
)


@flyte.trace
async def reason(goal: str, history: list[str]) -> dict:
    """The 'LLM' decides the next action. Mocked as a fixed policy; in production this
    is one chat-completions call and the decorator gives you a per-call trace + checkpoint."""
    if not any("calc:" in h for h in history):
        return {"action": "calculate", "input": goal}
    return {"action": "finish", "input": history[-1].removeprefix("calc:")}


@flyte.trace
async def act(action: str, action_input: str) -> str:
    """Tool dispatch. Tools can be plain code, other tasks, or a code sandbox."""
    if action == "calculate":
        # Untrusted-math tool: evaluate the expression in an isolated container,
        # NOT in the agent's process.
        calc = flyte.sandbox.create(
            name="calc-tool",
            code="result = float(eval(expression, {'__builtins__': {}}, {}))",
            inputs={"expression": str},
            outputs={"result": float},
        )
        value = await calc.run.aio(expression=action_input)
        return f"calc:{value}"
    raise ValueError(f"unknown tool {action}")


@agent_env.task
async def react_agent(goal: str, max_steps: int = 5) -> str:
    history: list[str] = []
    for _ in range(max_steps):
        decision = await reason(goal, history)             # traced checkpoint
        if decision["action"] == "finish":
            return decision["input"]
        history.append(await act(decision["action"], decision["input"]))  # traced
    return json.dumps({"error": "max_steps reached", "history": history})


run = await flyte.run.aio(react_agent, goal="(12 + 8) * 3")
print(run.url)
await run.wait.aio()
run.outputs()

Open the run URL: the `reason`/`act` calls appear as traced spans with their inputs/outputs,
and the calculator executed as its own sandboxed action. If the pod died mid-loop, the retry
would resume from the last completed trace instead of re-running the earlier LLM calls.

## 5. The rest of the agent stack

Everything else an agent needs maps to something this series already covered:

| Agent need | Union feature | Covered in |
|---|---|---|
| Human-in-the-loop approval | `flyte.new_condition` pauses the loop for a typed signal | [02](./02-parallelism-and-resilience.ipynb) §6 |
| Low-latency agent steps | `ReusePolicy` warm pools (model/clients stay loaded) | [04](./04-reusable-containers.ipynb) |
| Parallel sub-agents / plan-and-execute | `asyncio.gather` fan-out — each sub-task its own container | [02](./02-parallelism-and-resilience.ipynb) §1 |
| Chat UI / agent as a service | Apps (`AppEnvironment`, FastAPI/WebSocket) | [07](./07-apps-and-inference.ipynb) |
| LLM API keys | `flyte.Secret` | [05](./05-gcp-data-and-integrations.ipynb) §3 |
| Self-hosted models for agents | vLLM app (OpenAI-compatible) | [07](./07-apps-and-inference.ipynb) §3 |

Beyond hand-rolled loops, the docs' **Build an agent** section covers the batteries-included
*Flyte Agent harness* (tool registry, MCP servers, memory, HITL), the *agent chat UI*, and
integrations that run **LangGraph / PydanticAI / OpenAI Agents SDK** agents on Union with
durability added underneath.

## Further reading

- Union docs → User guide → **Sandboxing** (`code-sandboxing`, `code-mode`,
  `workflow-sandboxing-flyte`) and **Build an agent**
- Back to the series index: [README](./README.md)

> 💬 **Discuss:** which agent use cases are on your roadmap, and where would programmatic tool calling cut the most tokens/latency compared with sequential tool calling as you run it today?